In [5]:
#download step 1 dataset from huggingface
import csv
import json
from tqdm import tqdm
from datasets import load_dataset, Dataset
import os

if not os.path.exists('./filtered_data'):
    os.makedirs('./filtered_data')

#download step 1 dataset from huggingface
step1_data = load_dataset('talzoomanzoo/0514-step1-full-raw', split='train')

with open('./filtered_data/step1_data.json', 'w') as f:
    step1 = []
    for item in step1_data:
        step1.append({
            'id': item['id'],
            'year': item['year'],
            'problem_number': item['problem_number'],
            'Question': item['Question'],
            'answer': item['answer'],
            'Output_0': item['Output_0'],
            'output_tokens_0': item['output_tokens_0'],
            'Pred_Answer_0': item['Pred_Answer_0'],
            'Metrics_0': item['Metrics_0'],
            'Output_1': item['Output_1'],
            'output_tokens_1': item['output_tokens_1'],
            'Pred_Answer_1': item['Pred_Answer_1'],
            'Metrics_1': item['Metrics_1'],
            'Output_2': item['Output_2'],
            'output_tokens_2': item['output_tokens_2'],
            'Pred_Answer_2': item['Pred_Answer_2'],
            'Metrics_2': item['Metrics_2'],
            'Output_3': item['Output_3'],
            'output_tokens_3': item['output_tokens_3'],
            'Pred_Answer_3': item['Pred_Answer_3'],
            'Metrics_3': item['Metrics_3'],
            'Output_4': item['Output_4'],
            'output_tokens_4': item['output_tokens_4'],
            'Pred_Answer_4': item['Pred_Answer_4'],
            'Metrics_4': item['Metrics_4'],
            'per_question_mean_validity': item['per_question_mean_validity']
    })
    json.dump(step1, f, ensure_ascii=False, indent=4)

In [ ]:
#full step 2 data
import json
with open('/home/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    full_data = json.load(f)
len(full_data)

23325

In [7]:
#C. RFT for z where y = true
import json

with open('/home/hojinkim/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.28,22:10-5.json', 'r') as f:
    dpo_data = json.load(f)

dpo_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

# First, separate data into math_equal=True and math_equal=False groups
true_examples = []
false_examples = []

for data in dpo_data:
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    entry = {
        'idx': len(true_examples if has_math_equal else false_examples) + 1,
        'id': data.get('new_id', None),
        'input_x': '',
        'ground_truth': '',
        'output_z': '',
        'output_y': '',
        'metrics': []
    }

    # Collect all matching fields
    for key in data:
        if key.startswith('Question_'):
            entry['input_x'] = extract_x(data[key])
            entry['output_z'] = "<think>" + extract_z(data[key])
        elif key.startswith('Answer_'):
            entry['ground_truth'] = data[key]
        elif key.startswith('Output_'):
            entry['output_y'] = "</think>" + data[key]
        elif key.startswith('Metrics_'):
            entry['metrics'].append(data[key])
    
    if has_math_equal:
        true_examples.append(entry)
    else:
        false_examples.append(entry)

# Create pairs of examples
count = 0
for true_ex in true_examples:
    for false_ex in false_examples:
        if true_ex['input_x'] == false_ex['input_x']:  # Only pair examples with same input
            count += 1
            dpo_entry = {
                'idx': count,
                'id': f"{true_ex['id']}_{false_ex['id']}",
                'input_x': true_ex['input_x'],
                'ground_truth': true_ex['ground_truth'],
                'output_z': true_ex['output_z'],
                'chosen_y': true_ex['output_y'],
                'rejected_y': false_ex['output_y'],
                'metrics': true_ex['metrics'] + false_ex['metrics']
            }
            dpo_y_data.append(dpo_entry)

with open('0514.dpo_y_data.json', 'w') as f:
    json.dump(dpo_y_data, f, ensure_ascii=False, indent=4)
print(f"Created {len(dpo_y_data)} DPO pairs")
print(f"Total true examples: {len(true_examples)}")
print(f"Total false examples: {len(false_examples)}")

Created 109084 DPO pairs
Total true examples: 9481
Total false examples: 13844


In [8]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(dpo_y_data)
dataset.push_to_hub('talzoomanzoo/0514-dpo-y-data', private=False)

Uploading the dataset shards: 100%|██████████| 5/5 [00:11<00:00,  2.35s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0514-dpo-y-data/commit/682223824d02d20fff38532fa8bfacb9fcabf23c', commit_message='Upload dataset', commit_description='', oid='682223824d02d20fff38532fa8bfacb9fcabf23c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0514-dpo-y-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0514-dpo-y-data'), pr_revision=None, pr_num=None)